# 08 - RQ2: the 2x2x2x2 factorial
Run 1 (after 03): TF-IDF/LR, 16 cells x 3 seeds. Run 2 (after 05): best encoder, 6 cells x 3 seeds.
Dual evaluation: each cell's own test rows, and the clean test rows.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import leakage, models, plots
params = io_utils.read_json(config.SELECTION_LOCK)["models"]["tfidf-lr"]["params"]
clean_test = X[split == "test"]
rows = []
for cell in leakage.all_cells():
    for seed in config.SEEDS_FACTORIAL:
        Xc, yc, sc = labels.get_xy(df, cell, seed=seed)
        with evaluate.log_run("tfidf-lr", seed, cell.name, is_synthetic=SYN):
            r = models.tfidf_lr(Xc, yc, sc, seed=seed, **params)
        own = evaluate.test_metrics("tfidf-lr", yc[sc == "test"], r["scores"]["test"], is_synthetic=SYN)
        p_clean = r["model"].predict_proba(r["vectorizer"].transform(clean_test["text"]))[:, 1]
        cln = evaluate.test_metrics("tfidf-lr", y[clean_test.index], p_clean, is_synthetic=SYN)
        rows.append({**{f: getattr(cell, f) for f in leakage.FACTORS}, "cell": cell.name, "seed": seed,
                     **{f"own_{k}": v for k, v in own.items()}, **{f"clean_{k}": v for k, v in cln.items()}})
res = pd.DataFrame(rows)
evaluate.save_table(res, "table_rq2_tfidf_runs", is_synthetic=SYN)

In [ ]:
eff = leakage.factorial_effects(res.rename(columns={"own_lift": "lift"}), "lift")
evaluate.save_table(eff, "table_rq2_tfidf_effects_lift", is_synthetic=SYN)
plots.save_figure(plots.inflation_bars(eff), "fig_rq2_tfidf", None, SYN)
eff

Run 2: replace `models.tfidf_lr` with `models.finetune_encoder(..., config.ENCODERS[best])` over `leakage.encoder_cells()`.